In [ ]:
%load_ext autoreload
%autoreload 2


import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.linalg import expm

from pendulibrary.integrate import integrate_state
from pendulibrary.continuation import dc_underconstrained, adaptive_cont
from pendulibrary.common_targetters import single_fixed
from pendulibrary.common import get_A_raw

int_tol = 1e-14

In [ ]:
Lr = 2.0
Mr = 0.5
Xref = np.array([np.pi, 0.0, 0.0, 0.0])

In [ ]:
# both frequencies
A = get_A_raw(Xref, Lr, Mr)
eigs = np.linalg.eigvals(A)
print(eigs)
w = np.abs(eigs[-1].imag)

In [ ]:
T = 2 * np.pi / w
Phi = expm(A * T)
eigs = np.linalg.eig(Phi)
vals = eigs.eigenvalues
e1, e2 = eigs.eigenvectors[:, np.abs(vals - 1) < 1e-10].T

In [ ]:
force_0 = 0 if np.abs(e1[0]) > 0 and np.abs(e2[0]) > 0 else 1
if np.iscomplex(e1[0]):
    x0 = np.imag(e1 / e1[force_0])
else:
    x0 = np.real(e1 / e1[force_0] - e2 / e2[force_0])

x0 /= np.linalg.norm(x0)
x0 *= 1e-5
x0 += Xref

In [ ]:
targ = single_fixed(
    ind_fixed=1, val_fixed=0.0, ind_no_enforce=2, Lr=Lr, Mr=Mr, int_tol=int_tol
)
func = targ.f_df_stm
X, dF, _ = dc_underconstrained(
    targ.get_X(x0, T), func, 1e-12, debug=True, max_step=1e-2
)
x0, T = targ.get_x0(X), targ.get_period(X)
tangent = np.linalg.svd(dF).Vh[-1]

In [ ]:
x0

In [ ]:
Xs, eigs, (DFs, tans) = adaptive_cont(X, func, tangent, 5e-6, 1e-6, 0.1, 1e-12, 6, 3)

In [ ]:
np.linalg.svd(DFs[-1]).S

In [ ]:
Xs

In [ ]:
x0, T = targ.get_x0(Xs[-1]), targ.get_period(Xs[-1])

ts, xs, _ = integrate_state(x0, T, Lr, Mr, 1e-14)

In [ ]:
plt.plot(ts, xs.T - xs[:, 0])